# YOLO11n SimAM-CA Split-Policy Sweep (RTX4090)

Derived from `[FROM_NHU]yolov11n_simam_augmentation.ipynb`.

This notebook compares YOLO11n and YOLO11n + SimAM + Coordinate Attention across:

1. `plain_random_image`
2. `stratified_random_image`
3. `stratified_grouped_specimen`

It runs each split policy across seeds `42`, `123`, and `3407`.

Default contract matches the teammate notebook direction:

- heavier explicit augmentation,
- Ultralytics Albumentations hook disabled,
- grouped dataset parser retained,
- SimAM-CA architecture patch retained,
- RTX4090-oriented batch/workers/cache defaults.

## Important Outputs To Download

- `reports/simam_ca_split_policy_summary.csv`
- `reports/simam_ca_split_policy_partial.csv`
- `reports/simam_ca_split_policy_paper_table.csv`
- `reports/split_manifests/*`
- each run folder's `weights/best.pt`
- each run folder's `results.csv`
- `exports/simam_ca_split_policy_results.zip`

In [ ]:
from enum import Enum
from pathlib import Path
import os

class RuntimeMode(str, Enum):
    LOCAL_RTX4090 = "local_rtx4090"
    COLAB = "colab"
    KAGGLE = "kaggle"

RUNTIME_MODE = RuntimeMode.LOCAL_RTX4090

SMOKE_RUN = False
RUN_BASELINE_MODEL = True
RUN_SIMAM_CA_MODEL = True
RUN_PACKAGE_EXPORT = True

SPLIT_POLICIES = [
    "plain_random_image",
    "stratified_random_image",
    "stratified_grouped_specimen",
]
SEEDS = [42, 123, 3407]

TRAIN_IMGSZ = 320 if SMOKE_RUN else 640
TRAIN_EPOCHS = 1 if SMOKE_RUN else 100
TRAIN_PATIENCE = 1 if SMOKE_RUN else 30
TRAIN_BATCH = 8 if SMOKE_RUN else 64
TRAIN_WORKERS = 0 if SMOKE_RUN else 12
CACHE_IMAGES = False if SMOKE_RUN else "ram"
DEVICE = 0
AMP = True
STRICT_DETERMINISM = False

DISABLE_ULTRALYTICS_ALBUMENTATIONS = True
ULTRALYTICS_GIT_REF = "v8.4.67"

ROBOFLOW_API_KEY_DIRECT = ""
ROBOFLOW_WORKSPACE = "lets-try-this"
ROBOFLOW_PROJECT = "shrimpdishandsegv2"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolo26"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
PREDICT_CONF_FOR_COUNT = 0.25

if RUNTIME_MODE == RuntimeMode.LOCAL_RTX4090:
    ROOT_DIR = Path.cwd() / "simam_ca_split_policy_rtx4090_run"
    LOCAL_ULTRALYTICS_DIR = ROOT_DIR / "ultralytics_src"
    RUNS_DIR = ROOT_DIR / "runs" / "segment"
elif RUNTIME_MODE == RuntimeMode.COLAB:
    ROOT_DIR = Path("/content/simam_ca_split_policy_rtx4090_run")
    LOCAL_ULTRALYTICS_DIR = Path("/content/ultralytics")
    RUNS_DIR = Path("/content/runs/segment")
elif RUNTIME_MODE == RuntimeMode.KAGGLE:
    ROOT_DIR = Path("/kaggle/working/simam_ca_split_policy_rtx4090_run")
    LOCAL_ULTRALYTICS_DIR = Path("/kaggle/working/ultralytics")
    RUNS_DIR = Path("/kaggle/working/runs/segment")
else:
    raise ValueError(RUNTIME_MODE)

DATASET_RAW_DIR = ROOT_DIR / "dataset_raw"
WORK_DIR = ROOT_DIR / "work"
REPORT_DIR = ROOT_DIR / "reports"
EXPORT_DIR = ROOT_DIR / "exports"
SPLIT_MANIFEST_DIR = REPORT_DIR / "split_manifests"
for d in [ROOT_DIR, WORK_DIR, REPORT_DIR, EXPORT_DIR, SPLIT_MANIFEST_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Runtime mode:", RUNTIME_MODE)
print("ROOT_DIR:", ROOT_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("Batch/workers/cache:", TRAIN_BATCH, TRAIN_WORKERS, CACHE_IMAGES)
print("Split policies:", SPLIT_POLICIES)
print("Seeds:", SEEDS)
print("Disable Ultralytics Albumentations hook:", DISABLE_ULTRALYTICS_ALBUMENTATIONS)

## Setup And Dataset Download

No Roboflow key is committed. Set `ROBOFLOW_API_KEY` in the environment or cloud secret.

In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("roboflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])
if importlib.util.find_spec("yaml") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "PyYAML"])

from roboflow import Roboflow

def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key:
            return key.strip()
    except Exception:
        pass
    return os.environ.get("ROBOFLOW_API_KEY", "").strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError("Missing Roboflow API key. Set ROBOFLOW_API_KEY or temporarily fill ROBOFLOW_API_KEY_DIRECT.")

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_RAW_DIR))
base_path = str(Path(dataset.location))
print("Downloaded dataset to:", base_path)

## Split Utilities

In [ ]:
import hashlib
import json
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml

SHRIMP_NAME_PATTERN = re.compile(
    r"^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$",
    re.IGNORECASE,
)

def normalize_roboflow_stem(stem):
    stem = re.sub(r"_jpg\.rf\.[a-f0-9]+$", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"_png\.rf\.[a-f0-9]+$", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"\.rf\.[a-f0-9]+$", "", stem, flags=re.IGNORECASE)
    return stem

def parse_image_identity(image_name):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        raise ValueError(f"Cannot parse shrimp identity from {image_name} -> {stem}")
    disease = match.group("disease")
    shrimp_id = match.group("shrimp_id")
    img_num = int(match.group("img_num"))
    group_key = f"{disease.lower()}::{shrimp_id}"
    return group_key, disease, shrimp_id, img_num

def image_files_in_split(dataset_dir, split):
    image_dir = Path(dataset_dir) / split / "images"
    if not image_dir.exists():
        return []
    return sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS], key=lambda p: p.name)

def ensure_split_dirs(dataset_dir):
    for split in ["train", "valid", "test"]:
        for sub in ["images", "labels"]:
            (Path(dataset_dir) / split / sub).mkdir(parents=True, exist_ok=True)

def label_for_image(image_path):
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"

def move_image_and_label(image_path, target_split, dataset_dir):
    dataset_dir = Path(dataset_dir)
    target_img_dir = dataset_dir / target_split / "images"
    target_lbl_dir = dataset_dir / target_split / "labels"
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)
    image_dst = target_img_dir / image_path.name
    label_src = label_for_image(image_path)
    label_dst = target_lbl_dir / f"{image_path.stem}.txt"
    if image_path.resolve() != image_dst.resolve():
        shutil.move(str(image_path), str(image_dst))
    if label_src.exists() and label_src.resolve() != label_dst.resolve():
        shutil.move(str(label_src), str(label_dst))
    elif not label_dst.exists():
        label_dst.write_text("")

def rebuild_pool_from_all_splits(dataset_dir):
    dataset_dir = Path(dataset_dir)
    ensure_split_dirs(dataset_dir)
    all_images = []
    for split in ["train", "valid", "test"]:
        all_images.extend(image_files_in_split(dataset_dir, split))
    for image_path in all_images:
        if image_path.parent.parent.name != "train":
            move_image_and_label(image_path, "train", dataset_dir)
    return image_files_in_split(dataset_dir, "train")

def remove_yolo_label_caches(root):
    for cache in Path(root).rglob("*.cache"):
        try:
            cache.unlink()
        except FileNotFoundError:
            pass

def disease_for_image_name(name):
    return parse_image_identity(name)[1]

def disease_for_group(filenames):
    counts = Counter(disease_for_image_name(name) for name in filenames)
    return counts.most_common(1)[0][0]

def split_counts(n):
    train_count = int(TRAIN_RATIO * n)
    val_count = int(VAL_RATIO * n)
    test_count = n - train_count - val_count
    if n >= 3:
        if val_count == 0:
            val_count = 1
        if test_count == 0:
            test_count = 1
        while train_count + val_count + test_count > n:
            train_count -= 1
    return train_count, val_count, test_count

def split_items(items, rng):
    items = list(items)
    rng.shuffle(items)
    train_count, val_count, _ = split_counts(len(items))
    return {
        "train": items[:train_count],
        "valid": items[train_count:train_count + val_count],
        "test": items[train_count + val_count:],
    }

def apply_plain_random_image_split(dataset_dir, seed):
    rng = random.Random(seed)
    images = rebuild_pool_from_all_splits(dataset_dir)
    split_map = split_items(images, rng)
    for split, paths in split_map.items():
        for p in paths:
            move_image_and_label(p, split, dataset_dir)
    remove_yolo_label_caches(dataset_dir)
    return save_split_manifest(dataset_dir, "plain_random_image", seed)

def apply_stratified_random_image_split(dataset_dir, seed):
    rng = random.Random(seed)
    images = rebuild_pool_from_all_splits(dataset_dir)
    strata = defaultdict(list)
    for p in images:
        strata[disease_for_image_name(p.name)].append(p)
    split_map = {"train": [], "valid": [], "test": []}
    for disease, paths in sorted(strata.items()):
        part = split_items(paths, rng)
        for split in split_map:
            split_map[split].extend(part[split])
        print(f"{disease}: {len(part['train'])}/{len(part['valid'])}/{len(part['test'])} image split")
    for split, paths in split_map.items():
        for p in paths:
            move_image_and_label(p, split, dataset_dir)
    remove_yolo_label_caches(dataset_dir)
    return save_split_manifest(dataset_dir, "stratified_random_image", seed)

def apply_stratified_grouped_specimen_split(dataset_dir, seed):
    rng = random.Random(seed)
    images = rebuild_pool_from_all_splits(dataset_dir)
    groups = defaultdict(list)
    for p in images:
        group_key, *_ = parse_image_identity(p.name)
        groups[group_key].append(p.name)
    strata = defaultdict(list)
    for group_key, names in groups.items():
        strata[disease_for_group(names)].append((group_key, names))
    split_groups = {"train": [], "valid": [], "test": []}
    for disease, group_items in sorted(strata.items()):
        group_items = sorted(group_items, key=lambda x: x[0])
        part = split_items(group_items, rng)
        for split in split_groups:
            split_groups[split].extend(part[split])
        print(f"{disease}: {len(part['train'])}/{len(part['valid'])}/{len(part['test'])} group split")
    name_to_split = {}
    for split, group_items in split_groups.items():
        for _, names in group_items:
            for name in names:
                name_to_split[name] = split
    for p in image_files_in_split(dataset_dir, "train"):
        move_image_and_label(p, name_to_split[p.name], dataset_dir)
    remove_yolo_label_caches(dataset_dir)
    return save_split_manifest(dataset_dir, "stratified_grouped_specimen", seed)

def group_overlap(dataset_dir, a, b):
    groups_a = {parse_image_identity(p.name)[0] for p in image_files_in_split(dataset_dir, a)}
    groups_b = {parse_image_identity(p.name)[0] for p in image_files_in_split(dataset_dir, b)}
    return len(groups_a & groups_b)

def label_stats(dataset_dir, split):
    label_dir = Path(dataset_dir) / split / "labels"
    image_dir = Path(dataset_dir) / split / "images"
    image_count = len([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]) if image_dir.exists() else 0
    labeled = healthy = instances = 0
    for label_path in sorted(label_dir.glob("*.txt")) if label_dir.exists() else []:
        lines = [x.strip() for x in label_path.read_text().splitlines() if x.strip()]
        if lines:
            labeled += 1
            instances += len(lines)
        else:
            healthy += 1
    return {"images": image_count, "labeled_images": labeled, "healthy_images": healthy, "instances": instances}

def save_split_manifest(dataset_dir, policy, seed):
    rows = []
    for split in ["train", "valid", "test"]:
        for p in image_files_in_split(dataset_dir, split):
            group_key, disease, shrimp_id, img_num = parse_image_identity(p.name)
            rows.append({"split": split, "image": p.name, "group_key": group_key, "disease": disease, "shrimp_id": shrimp_id, "img_num": img_num})
    manifest = pd.DataFrame(rows).sort_values(["split", "group_key", "image"])
    manifest_path = SPLIT_MANIFEST_DIR / f"{policy}_seed{seed}_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    payload = "\n".join(f"{r.split},{r.image},{r.group_key}" for r in manifest.itertuples(index=False))
    fingerprint = hashlib.sha256(payload.encode()).hexdigest()
    (SPLIT_MANIFEST_DIR / f"{policy}_seed{seed}_fingerprint.txt").write_text(fingerprint)
    summary_rows = []
    for split in ["train", "valid", "test"]:
        stats = label_stats(dataset_dir, split)
        summary_rows.append({"split": split, **stats, "specimens": int(manifest[manifest["split"] == split]["group_key"].nunique())})
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(SPLIT_MANIFEST_DIR / f"{policy}_seed{seed}_summary.csv", index=False)
    print("Split fingerprint:", fingerprint)
    print(summary)
    return {"manifest": str(manifest_path), "fingerprint": fingerprint, "summary": summary.to_dict(orient="records")}

def apply_split_policy(dataset_dir, policy, seed):
    if policy == "plain_random_image":
        return apply_plain_random_image_split(dataset_dir, seed)
    if policy == "stratified_random_image":
        return apply_stratified_random_image_split(dataset_dir, seed)
    if policy == "stratified_grouped_specimen":
        return apply_stratified_grouped_specimen_split(dataset_dir, seed)
    raise ValueError(policy)

## Patch Ultralytics With SimAM/CA Modules

In [ ]:
from pathlib import Path
import subprocess
import sys
import py_compile

ULTRA_DIR = LOCAL_ULTRALYTICS_DIR

if not ULTRA_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ultralytics/ultralytics.git", str(ULTRA_DIR)], check=True)
    if ULTRALYTICS_GIT_REF:
        subprocess.run(["git", "-C", str(ULTRA_DIR), "checkout", ULTRALYTICS_GIT_REF], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ULTRA_DIR)], check=True)

conv_py = ULTRA_DIR / "ultralytics/nn/modules/conv.py"
init_py = ULTRA_DIR / "ultralytics/nn/modules/__init__.py"
tasks_py = ULTRA_DIR / "ultralytics/nn/tasks.py"

for p in [conv_py, init_py, tasks_py]:
    backup = p.with_suffix(p.suffix + ".bak_all_attention_group_run")
    if not backup.exists():
        backup.write_text(p.read_text())
        print("Backup created:", backup)

attention_code = """
# =========================================================
# Custom attention modules for YOLO11 segmentation experiments
# =========================================================

class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (
            4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        c2 = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        if self.proj is not None:
            identity = self.proj(identity)
        return identity * a_h * a_w


class ECAAttention(nn.Module):
    def __init__(self, c1, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class CBAMAttention(nn.Module):
    def __init__(self, c1, c2=None, reduction=16, kernel_size=7):
        super().__init__()
        c2 = c1 if c2 is None else c2
        hidden = max(c1 // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(c1, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, c1, kernel_size=1, bias=False),
        )
        self.channel_sigmoid = nn.Sigmoid()
        assert kernel_size in (3, 7), "CBAM spatial kernel_size should be 3 or 7"
        padding = 3 if kernel_size == 7 else 1
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.spatial_sigmoid = nn.Sigmoid()
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        ca = self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x))
        x = x * self.channel_sigmoid(ca)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.cat([avg_out, max_out], dim=1)
        x = x * self.spatial_sigmoid(self.spatial_conv(sa))
        if self.proj is not None:
            x = self.proj(x)
        return x


class EMAAttention(nn.Module):
    def __init__(self, c1, c2=None, groups=8):
        super().__init__()
        c2 = c1 if c2 is None else c2
        groups = max(1, int(groups))
        groups = min(groups, c1)
        if c1 % groups != 0:
            valid_groups = [g for g in [8, 4, 2, 1] if c1 % g == 0]
            groups = valid_groups[0] if valid_groups else 1
        self.c1 = c1
        self.c2 = c2
        self.groups = groups
        self.group_channels = c1 // groups
        self.softmax = nn.Softmax(dim=-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.conv1x1 = nn.Conv2d(self.group_channels, self.group_channels, kernel_size=1, stride=1, padding=0)
        self.conv3x3 = nn.Conv2d(self.group_channels, self.group_channels, kernel_size=3, stride=1, padding=1)
        self.gn = nn.GroupNorm(self.group_channels, self.group_channels)
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        b, c, h, w = x.size()
        if c != self.c1:
            return x
        if c % self.groups != 0:
            return x
        group_x = x.reshape(b * self.groups, self.group_channels, h, w)
        x_h = group_x.mean(dim=3, keepdim=True)
        x_w = group_x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, self.group_channels, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, self.group_channels, h * w)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, self.group_channels, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, self.group_channels, h * w)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, h, w)
        out = (group_x * weights.sigmoid()).reshape(b, c, h, w)
        if self.proj is not None:
            out = self.proj(out)
        return out
"""

# =========================================================
# Patch conv.py
# =========================================================
text = conv_py.read_text()
text = text.replace(r'\"\"\"', '"""')

if "import torch\n" not in text:
    text = text.replace("import math\n", "import math\nimport torch\n") if "import math\n" in text else "import torch\n" + text

if "import torch.nn as nn\n" not in text and "from torch import nn\n" not in text:
    text = text.replace("import torch\n", "import torch\nimport torch.nn as nn\n")

for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
    token = f'"{module_name}",'
    if token not in text:
        if '"RepConv",\n    "SpatialAttention",' in text:
            text = text.replace(
                '    "RepConv",\n    "SpatialAttention",',
                f'    "RepConv",\n    "{module_name}",\n    "SpatialAttention",'
            )
        elif '"Concat",' in text:
            text = text.replace('"Concat",', f'"Concat",\n    "{module_name}",')

missing_any = any(
    f"class {m}(nn.Module):" not in text
    for m in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]
)
if missing_any:
    text = text.rstrip() + "\n" + attention_code + "\n"

conv_py.write_text(text)
print("Patched conv.py with attention modules")

try:
    py_compile.compile(str(conv_py), doraise=True)
    print("conv.py syntax OK")
except Exception as e:
    print("conv.py syntax error:")
    print(e)
    lines = conv_py.read_text().splitlines()
    line_no = getattr(getattr(e, "exc_value", None), "lineno", 1)
    start = max(0, line_no - 8)
    end = min(len(lines), line_no + 8)
    print(f"\n--- conv.py lines {start + 1} to {end} ---")
    for i in range(start, end):
        print(f"{i + 1}: {lines[i]}")
    raise

# =========================================================
# Patch __init__.py
# =========================================================
text = init_py.read_text()

if "from .conv import (" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        conv_import_block = text.split("from .conv import (", 1)[1].split(")", 1)[0]
        if module_name not in conv_import_block:
            if "    RepConv,\n    SpatialAttention," in text:
                text = text.replace(
                    "    RepConv,\n    SpatialAttention,",
                    f"    RepConv,\n    {module_name},\n    SpatialAttention,"
                )
            elif "    Concat," in text:
                text = text.replace("    Concat,", f"    Concat,\n    {module_name},")
            else:
                text = text.replace("from .conv import (", f"from .conv import (\n    {module_name},")
else:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        if f"from .conv import {module_name}" not in text:
            text += f"\nfrom .conv import {module_name}\n"

if "__all__" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        all_block = text.split("__all__", 1)[1]
        if f'"{module_name}"' not in all_block:
            if '"SemanticSegment",\n    "SpatialAttention",' in text:
                text = text.replace(
                    '    "SemanticSegment",\n    "SpatialAttention",',
                    f'    "SemanticSegment",\n    "{module_name}",\n    "SpatialAttention",'
                )
            elif '"Concat",' in text:
                text = text.replace('"Concat",', f'"Concat",\n    "{module_name}",')

init_py.write_text(text)
print("Patched __init__.py")

# =========================================================
# Patch tasks.py
# =========================================================
text = tasks_py.read_text()

if "from ultralytics.nn.modules import (" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        modules_import_block = text.split("from ultralytics.nn.modules import (", 1)[1].split(")", 1)[0]
        if module_name not in modules_import_block:
            if "    Segment,\n    Segment26," in text:
                text = text.replace(
                    "    Segment,\n    Segment26,",
                    f"    Segment,\n    Segment26,\n    {module_name},"
                )
            elif "    Concat," in text:
                text = text.replace("    Concat,", f"    Concat,\n    {module_name},")
            else:
                text = text.replace(
                    "from ultralytics.nn.modules import (",
                    f"from ultralytics.nn.modules import (\n    {module_name},"
                )

base_start = text.find("base_modules = frozenset")
repeat_start = text.find("repeat_modules = frozenset", base_start)
if base_start == -1 or repeat_start == -1:
    raise RuntimeError("Could not locate base_modules block in tasks.py")

def add_to_base_modules(text, module_name):
    base_start = text.find("base_modules = frozenset")
    repeat_start = text.find("repeat_modules = frozenset", base_start)
    base_block = text[base_start:repeat_start]
    if module_name in base_block:
        print(f"{module_name} already exists in base_modules")
        return text
    pattern1 = "base_modules = frozenset(\n        {"
    pattern2 = "base_modules = frozenset({"
    if pattern1 in text:
        text = text.replace(pattern1, f"base_modules = frozenset(\n        {{\n            {module_name},")
        print(f"Added {module_name} to base_modules using pattern1")
    elif pattern2 in text:
        text = text.replace(pattern2, f"base_modules = frozenset({{\n            {module_name},")
        print(f"Added {module_name} to base_modules using pattern2")
    else:
        raise RuntimeError(f"Could not insert {module_name} into base_modules.")
    return text

for module_name in ["CoordAtt", "CBAMAttention", "EMAAttention"]:
    text = add_to_base_modules(text, module_name)

# ECA and SimAM keep channel count; parse manually.
if "elif m is ECAAttention:" not in text:
    simam_anchor = """elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]"""
    eca_branch = """elif m is ECAAttention:
            c2 = ch[f]
            if len(args) >= 2:
                k_size = args[1]
            elif len(args) == 1:
                k_size = args[0]
            else:
                k_size = 3
            args = [c2, k_size]
        elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]"""
    if simam_anchor in text:
        text = text.replace(simam_anchor, eca_branch)
    else:
        detect_anchor = """elif m in frozenset(
            {
                Detect,"""
        eca_and_simam_branch = """elif m is ECAAttention:
            c2 = ch[f]
            if len(args) >= 2:
                k_size = args[1]
            elif len(args) == 1:
                k_size = args[0]
            else:
                k_size = 3
            args = [c2, k_size]
        elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]
        """
        if detect_anchor in text:
            text = text.replace(detect_anchor, eca_and_simam_branch + detect_anchor)
        else:
            raise RuntimeError("Could not find insertion point for ECAAttention/SimAM parse branch.")

if "elif m is SimAM:" not in text:
    detect_anchor = """elif m in frozenset(
            {
                Detect,"""
    simam_branch = """elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]
        """
    if detect_anchor in text:
        text = text.replace(detect_anchor, simam_branch + detect_anchor)
    else:
        raise RuntimeError("Could not find insertion point for SimAM parse branch.")

tasks_py.write_text(text)
print("Patched tasks.py")

print("Attention registration complete.")
print("IMPORTANT: Restart runtime/kernel before creating YOLO models.")

In [ ]:
# =========================================================
# Force using local patched Ultralytics repo
# =========================================================
import sys
import subprocess
from pathlib import Path

ULTRA_DIR = LOCAL_ULTRALYTICS_DIR

if not ULTRA_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/ultralytics.git", str(ULTRA_DIR)],
        check=True
    )

if ULTRALYTICS_GIT_REF:
    subprocess.run(["git", "-C", str(ULTRA_DIR), "checkout", ULTRALYTICS_GIT_REF], check=False)

# Install local editable repo
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(ULTRA_DIR)],
    check=True
)

# Put local repo at the front of Python import path
sys.path.insert(0, str(ULTRA_DIR))

# Clear previously imported pip ultralytics modules
for module_name in list(sys.modules.keys()):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        del sys.modules[module_name]

import ultralytics
from ultralytics import YOLO

print("Using Ultralytics from:")
print(ultralytics.__file__)

assert str(ULTRA_DIR) in ultralytics.__file__, (
    "Still importing wrong Ultralytics version. "
    f"Expected local repo: {ULTRA_DIR}, got: {ultralytics.__file__}"
)

In [ ]:
from pathlib import Path

ULTRA_DIR = LOCAL_ULTRALYTICS_DIR
MODEL_CFG_DIR = ULTRA_DIR / "ultralytics/cfg/models/11"
MODEL_CFG_DIR.mkdir(parents=True, exist_ok=True)

BASE_YAML_TEMPLATE = """nc: 2
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

{attention_layers}
"""

YAML_LAYER_DEFS = {
    "simam": ("yolo11n-seg-simam-head.yaml", """  # SimAM before Segment head.
  - [16, 1, SimAM, []]
  - [19, 1, SimAM, []]
  - [22, 1, SimAM, []]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "ca": ("yolo11n-seg-ca-head.yaml", """  # Coordinate Attention before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [19, 1, CoordAtt, [512, 32]]
  - [22, 1, CoordAtt, [1024, 32]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "eca": ("yolo11n-seg-eca-head.yaml", """  # ECA before Segment head.
  - [16, 1, ECAAttention, [3]]
  - [19, 1, ECAAttention, [3]]
  - [22, 1, ECAAttention, [3]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "cbam": ("yolo11n-seg-cbam-head.yaml", """  # CBAM before Segment head.
  - [16, 1, CBAMAttention, [256, 16, 7]]
  - [19, 1, CBAMAttention, [512, 16, 7]]
  - [22, 1, CBAMAttention, [1024, 16, 7]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "ema": ("yolo11n-seg-ema-head.yaml", """  # EMA before Segment head.
  - [16, 1, EMAAttention, [256, 8]]
  - [19, 1, EMAAttention, [512, 8]]
  - [22, 1, EMAAttention, [1024, 8]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "simam_ca": ("yolo11n-seg-simam-ca-head.yaml", """  # CA -> SimAM before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, SimAM, []]

  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, SimAM, []]

  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "eca_simam": ("yolo11n-seg-eca-simam-head.yaml", """  # ECA -> SimAM before Segment head.
  - [16, 1, ECAAttention, [3]]
  - [23, 1, SimAM, []]

  - [19, 1, ECAAttention, [3]]
  - [25, 1, SimAM, []]

  - [22, 1, ECAAttention, [3]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "ca_ema": ("yolo11n-seg-ca-ema-head.yaml", """  # CA -> EMA before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, EMAAttention, [256, 8]]

  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, EMAAttention, [512, 8]]

  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, EMAAttention, [1024, 8]]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "cbam_simam": ("yolo11n-seg-cbam-simam-head.yaml", """  # CBAM -> SimAM before Segment head.
  - [16, 1, CBAMAttention, [256, 16, 7]]
  - [23, 1, SimAM, []]

  - [19, 1, CBAMAttention, [512, 16, 7]]
  - [25, 1, SimAM, []]

  - [22, 1, CBAMAttention, [1024, 16, 7]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "ema_simam_ca": ("yolo11n-seg-ema-simam-ca-head.yaml", """  # CA -> EMA -> SimAM before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, EMAAttention, [256, 8]]
  - [24, 1, SimAM, []]

  - [19, 1, CoordAtt, [512, 32]]
  - [26, 1, EMAAttention, [512, 8]]
  - [27, 1, SimAM, []]

  - [22, 1, CoordAtt, [1024, 32]]
  - [29, 1, EMAAttention, [1024, 8]]
  - [30, 1, SimAM, []]

  - [[25, 28, 31], 1, Segment, [nc, 32, 256]]"""),
}

MODEL_YAML_PATHS = {}
for key, (yaml_name, layers) in YAML_LAYER_DEFS.items():
    yaml_text = BASE_YAML_TEMPLATE.format(attention_layers=layers)
    yaml_path = MODEL_CFG_DIR / yaml_name
    yaml_path.write_text(yaml_text)
    MODEL_YAML_PATHS[key] = str(yaml_path)
    print("Created:", key, "->", yaml_path)

## Experiment Definitions

In [ ]:
EXPERIMENT_MODELS = []
if RUN_BASELINE_MODEL:
    EXPERIMENT_MODELS.append({"key": "baseline", "name": "Baseline YOLO11n-seg", "model_type": "baseline", "model": "yolo11n-seg.pt"})
if RUN_SIMAM_CA_MODEL:
    EXPERIMENT_MODELS.append({"key": "simam_ca", "name": "YOLO11n-seg + SimAM + CA", "model_type": "attention", "yaml": MODEL_YAML_PATHS["simam_ca"]})

TEAMMATE_HEAVY_TRAIN_ARGS = {
    "auto_augment": None,
    "erasing": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.3,
    "hsv_h": 0.05,
    "hsv_s": 0.50,
    "hsv_v": 0.40,
    "degrees": 10.0,
    "translate": 0.10,
    "scale": 0.50,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}

print("Enabled models:")
for m in EXPERIMENT_MODELS:
    print("-", m["key"], m["name"])
print("Train args:", TEAMMATE_HEAVY_TRAIN_ARGS)

## Evaluation Utilities

In [ ]:
import csv
import gc
import math
import os
import time
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if STRICT_DETERMINISM:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as e:
            print("Deterministic warning:", e)
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False

def configure_ultralytics_albumentations():
    if not DISABLE_ULTRALYTICS_ALBUMENTATIONS:
        print("Ultralytics Albumentations hook left enabled.")
        return
    try:
        import ultralytics.data.augment as yolo_aug
        if hasattr(yolo_aug, "Albumentations"):
            class NoAlbumentations:
                def __init__(self, *args, **kwargs):
                    pass
                def __call__(self, labels):
                    return labels
            yolo_aug.Albumentations = NoAlbumentations
            print("Disabled default Ultralytics Albumentations hook.")
    except Exception as e:
        print("Could not disable Albumentations hook:", e)

def copy_dataset_for_run(policy, seed, model_key):
    dst = WORK_DIR / f"{policy}_seed{seed}_{model_key}" / "dataset"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(base_path, dst, ignore=shutil.ignore_patterns("*.cache", "runs"))
    remove_yolo_label_caches(dst)
    return dst

def write_data_yaml(dataset_dir):
    dataset_dir = Path(dataset_dir)
    source_yaml = Path(base_path) / "data.yaml"
    content = yaml.safe_load(source_yaml.read_text()) if source_yaml.exists() else {"names": ["BG", "WSSV"], "nc": 2}
    content["path"] = str(dataset_dir)
    content["train"] = str(dataset_dir / "train" / "images")
    content["val"] = str(dataset_dir / "valid" / "images")
    content["test"] = str(dataset_dir / "test" / "images")
    yaml_path = dataset_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(content, sort_keys=False))
    return yaml_path

def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for image_path in sorted(src_images.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label_path = src_labels / f"{image_path.stem}.txt"
        is_labeled = label_path.exists() and bool(label_path.read_text().strip())
        if is_labeled == want_labeled:
            shutil.copy2(image_path, dst_images / image_path.name)
            shutil.copy2(label_path, dst_labels / label_path.name)
            copied += 1
    return copied

def make_eval_dataset(src_dataset, state_name, want_labeled):
    dst = Path(src_dataset).parent / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for split in ["valid", "test"]:
        copy_split_by_label_state(src_dataset, dst, split, want_labeled)
    yaml_path = write_data_yaml(dst)
    return dst, yaml_path

def metric_value(metrics, dotted_path):
    value = metrics
    for part in dotted_path.split("."):
        value = getattr(value, part)
    try:
        return float(value)
    except Exception:
        return float("nan")

def read_yolo_label_count(label_path):
    if not Path(label_path).exists():
        return 0
    return len([x for x in Path(label_path).read_text().splitlines() if x.strip()])

def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted([p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
    if not image_paths:
        return {"images": 0, "gt_total": 0, "pred_mask_total": 0, "mask_count_mae": 0.0, "disease_mask_miss_rate": 0.0}
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    gt_total = pred_total = missed = over = under = 0
    abs_errors = []
    for image_path, result in zip(image_paths, results):
        gt = read_yolo_label_count(Path(labels_dir) / f"{image_path.stem}.txt")
        pred = 0 if result.masks is None else len(result.masks)
        gt_total += gt
        pred_total += pred
        abs_errors.append(abs(pred - gt))
        if gt > 0 and pred == 0:
            missed += 1
        if pred > gt:
            over += 1
        if pred < gt:
            under += 1
    n = len(image_paths)
    return {
        "images": n,
        "gt_total": gt_total,
        "pred_mask_total": pred_total,
        "mask_count_mae": float(np.mean(abs_errors)) if abs_errors else 0.0,
        "disease_mask_miss_rate": missed / n if n else 0.0,
        "over_pred_image_rate": over / n if n else 0.0,
        "under_pred_image_rate": under / n if n else 0.0,
    }

def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted([p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
    if not image_paths:
        return {"healthy_images": 0, "healthy_mask_fp_rate": 0.0, "healthy_fp_masks_total": 0, "healthy_fp_masks_per_image": 0.0, "healthy_avg_fp_confidence": 0.0}
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    fp_images = fp_masks = 0
    confidences = []
    for result in results:
        count = 0 if result.masks is None else len(result.masks)
        if count > 0:
            fp_images += 1
            fp_masks += count
            if result.boxes is not None and result.boxes.conf is not None:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_mask_fp_rate": fp_images / n,
        "healthy_fp_masks_total": fp_masks,
        "healthy_fp_masks_per_image": fp_masks / n,
        "healthy_avg_fp_confidence": float(np.mean(confidences)) if confidences else 0.0,
    }

def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    if df.empty:
        return {"epochs_ran": 0}
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    last = df.iloc[-1]
    out = {"epochs_ran": int(len(df))}
    if mask_col in df.columns:
        best_idx = df[mask_col].astype(float).idxmax()
        best = df.loc[best_idx]
        out.update({
            "best_epoch_by_mask_map50": int(best.get("epoch", best_idx + 1)),
            "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
            "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
            "last_val_mask_map50": float(last.get(mask_col, float("nan"))),
            "last_val_mask_map50_95": float(last.get("metrics/mAP50-95(M)", float("nan"))),
        })
    return out

def make_model(exp):
    if exp["model_type"] == "baseline":
        return YOLO(exp["model"])
    model = YOLO(exp["yaml"])
    try:
        model.load("yolo11n-seg.pt")
        print(f"Loaded pretrained weights for {exp['key']}")
    except Exception as e:
        print(f"Pretrained load warning for {exp['key']}:", e)
    return model

## Run Split-Policy / Seed / Model Matrix

In [ ]:
configure_ultralytics_albumentations()
all_rows = []
partial_csv = REPORT_DIR / "simam_ca_split_policy_partial.csv"

for seed in SEEDS:
    set_seed(seed)
    for split_policy in SPLIT_POLICIES:
        for exp in EXPERIMENT_MODELS:
            print("\n" + "=" * 120)
            print(f"Seed={seed} | split_policy={split_policy} | model={exp['key']}")
            print("=" * 120)

            dataset_dir = copy_dataset_for_run(split_policy, seed, exp["key"])
            split_info = apply_split_policy(dataset_dir, split_policy, seed)
            yaml_path = write_data_yaml(dataset_dir)
            labeled_eval_dir, labeled_eval_yaml = make_eval_dataset(dataset_dir, "labeled_only", want_labeled=True)
            healthy_eval_dir, healthy_eval_yaml = make_eval_dataset(dataset_dir, "healthy_only", want_labeled=False)

            split_stats = {s: label_stats(dataset_dir, s) for s in ["train", "valid", "test"]}
            train_test_overlap = group_overlap(dataset_dir, "train", "test")
            valid_test_overlap = group_overlap(dataset_dir, "valid", "test")
            print("Split stats:", split_stats)
            print("train/test group overlap:", train_test_overlap)
            print("valid/test group overlap:", valid_test_overlap)

            run_name = f"simam_ca_split_{split_policy}_seed{seed}_{exp['key']}"
            yolo = make_model(exp)
            start = time.time()
            yolo.train(
                data=str(yaml_path),
                task="segment",
                imgsz=TRAIN_IMGSZ,
                epochs=TRAIN_EPOCHS,
                batch=TRAIN_BATCH,
                patience=TRAIN_PATIENCE,
                seed=seed,
                deterministic=STRICT_DETERMINISM,
                workers=TRAIN_WORKERS,
                cache=CACHE_IMAGES,
                device=DEVICE,
                amp=AMP,
                project=str(RUNS_DIR),
                name=run_name,
                exist_ok=True,
                pretrained=True,
                plots=True,
                verbose=True,
                **TEAMMATE_HEAVY_TRAIN_ARGS,
            )
            train_time_min = (time.time() - start) / 60.0

            run_path = RUNS_DIR / run_name
            best_path = run_path / "weights" / "best.pt"
            best_model = YOLO(str(best_path))

            full_val = best_model.val(data=str(yaml_path), split="val", imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
            full_test = best_model.val(data=str(yaml_path), split="test", imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
            labeled_val = best_model.val(data=str(labeled_eval_yaml), split="val", imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
            labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=TRAIN_IMGSZ, plots=False, verbose=False)

            labeled_val_count = count_prediction_errors(best_model, labeled_eval_dir / "valid" / "images", labeled_eval_dir / "valid" / "labels")
            labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / "test" / "images", labeled_eval_dir / "test" / "labels")
            healthy_val_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "valid" / "images")
            healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images")

            labeled_val_mask_map50 = metric_value(labeled_val, "seg.map50")
            labeled_test_mask_map50 = metric_value(labeled_test, "seg.map50")
            healthy_aware_val = labeled_val_mask_map50 - COUNT_PENALTY_WEIGHT * labeled_val_count["mask_count_mae"] - DISEASE_MISS_PENALTY_WEIGHT * labeled_val_count["disease_mask_miss_rate"] - HEALTHY_FP_PENALTY_WEIGHT * healthy_val_fp["healthy_mask_fp_rate"]
            healthy_aware_test = labeled_test_mask_map50 - COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"] - DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_mask_miss_rate"] - HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]

            row = {
                "seed": seed,
                "split_policy": split_policy,
                "experiment": exp["key"],
                "name": exp["name"],
                "model_type": exp["model_type"],
                "run_name": run_name,
                "run_path": str(run_path),
                "best_pt": str(best_path),
                "split_fingerprint": split_info["fingerprint"],
                "train_time_min": round(train_time_min, 2),
                "train_images": split_stats["train"]["images"],
                "valid_images": split_stats["valid"]["images"],
                "test_images": split_stats["test"]["images"],
                "train_labeled_images": split_stats["train"]["labeled_images"],
                "valid_labeled_images": split_stats["valid"]["labeled_images"],
                "test_labeled_images": split_stats["test"]["labeled_images"],
                "train_healthy_images": split_stats["train"]["healthy_images"],
                "valid_healthy_images": split_stats["valid"]["healthy_images"],
                "test_healthy_images": split_stats["test"]["healthy_images"],
                "train_test_group_overlap": train_test_overlap,
                "valid_test_group_overlap": valid_test_overlap,
                "full_val_mask_map50": metric_value(full_val, "seg.map50"),
                "full_val_mask_map50_95": metric_value(full_val, "seg.map"),
                "full_test_mask_map50": metric_value(full_test, "seg.map50"),
                "full_test_mask_map50_95": metric_value(full_test, "seg.map"),
                "labeled_val_mask_map50": labeled_val_mask_map50,
                "labeled_val_mask_map50_95": metric_value(labeled_val, "seg.map"),
                "labeled_test_mask_map50": labeled_test_mask_map50,
                "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),
                "healthy_val_mask_fp_rate": healthy_val_fp["healthy_mask_fp_rate"],
                "healthy_val_fp_masks_per_image": healthy_val_fp["healthy_fp_masks_per_image"],
                "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
                "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
                "labeled_val_mask_count_mae": labeled_val_count["mask_count_mae"],
                "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
                "labeled_val_disease_mask_miss_rate": labeled_val_count["disease_mask_miss_rate"],
                "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
                "healthy_aware_labeled_val_mask_map50": healthy_aware_val,
                "healthy_aware_labeled_test_mask_map50": healthy_aware_test,
                "disable_ultralytics_albumentations": DISABLE_ULTRALYTICS_ALBUMENTATIONS,
                "train_args": json.dumps(TEAMMATE_HEAVY_TRAIN_ARGS, sort_keys=True),
            }
            row.update(read_best_epoch_from_results(run_path))
            all_rows.append(row)
            df = pd.DataFrame(all_rows)
            df.to_csv(partial_csv, index=False)
            display(df.tail(1))
            print("Partial saved:", partial_csv)

            del yolo, best_model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_rows)
summary_csv = REPORT_DIR / "simam_ca_split_policy_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved summary:", summary_csv)
display(summary_df)

paper_cols = [
    "seed", "split_policy", "experiment", "train_test_group_overlap",
    "full_test_mask_map50", "labeled_test_mask_map50", "labeled_test_mask_map50_95",
    "healthy_test_mask_fp_rate", "healthy_test_fp_masks_per_image",
    "labeled_test_disease_mask_miss_rate", "labeled_test_mask_count_mae",
    "healthy_aware_labeled_val_mask_map50", "healthy_aware_labeled_test_mask_map50",
    "best_pt",
]
paper_df = summary_df[[c for c in paper_cols if c in summary_df.columns]].copy()
paper_csv = REPORT_DIR / "simam_ca_split_policy_paper_table.csv"
paper_df.to_csv(paper_csv, index=False)
print("Saved paper table:", paper_csv)
display(paper_df)

## Package Export

In [ ]:
if RUN_PACKAGE_EXPORT:
    import zipfile
    export_zip = EXPORT_DIR / "simam_ca_split_policy_results.zip"
    with zipfile.ZipFile(export_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in REPORT_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(ROOT_DIR))
        for run_dir in RUNS_DIR.glob("simam_ca_split_*"):
            for rel in ["results.csv", "args.yaml", "results.png"]:
                p = run_dir / rel
                if p.exists():
                    zf.write(p, p.relative_to(ROOT_DIR))
            for weights_name in ["best.pt", "last.pt"]:
                p = run_dir / "weights" / weights_name
                if p.exists():
                    zf.write(p, p.relative_to(ROOT_DIR))
        zf.writestr("README_RESULTS_PACKAGE.txt", "YOLO11n SimAM-CA split-policy sweep results.\\n")
    print("Export zip:", export_zip)

print("Download these files:")
print("-", REPORT_DIR / "simam_ca_split_policy_summary.csv")
print("-", REPORT_DIR / "simam_ca_split_policy_partial.csv")
print("-", REPORT_DIR / "simam_ca_split_policy_paper_table.csv")
print("-", SPLIT_MANIFEST_DIR)
print("-", EXPORT_DIR / "simam_ca_split_policy_results.zip")